In [1]:
import json
import hashlib
import sys
from urllib.request import Request, build_opener, ProxyHandler

MODELO = "qwen3.5:4b-q4_K_M"
API_LOCAL = "http://127.0.0.1:11434"

# Archivos comunes del experimento
with open("../casos_comparacion_v1.json", "rb") as f:
    bytes_casos = f.read()

with open("../configuracion_v3.json", "rb") as f:
    bytes_config = f.read()

casos = json.loads(bytes_casos)
configuracion = json.loads(bytes_config)

# Comprobar integridad
sha_casos = hashlib.sha256(bytes_casos).hexdigest()
sha_config = hashlib.sha256(bytes_config).hexdigest()

assert sha_casos == configuracion["cases_sha256"]
assert sha_config == (
    "1c23edad394c7f555f05f2a464440e80cefa1023cb9026593e39eeebacd918af"
)

assert [c["case_id"] for c in casos] == [
    "S-005", "S-006", "S-007",
    "S-008", "S-009", "S-010"
]

assert configuracion["api_url"] == API_LOCAL + "/api/chat"

# Conexión local, sin proxy
cliente_local = build_opener(ProxyHandler({}))

with cliente_local.open(API_LOCAL + "/api/version", timeout=10) as f:
    version_ollama = json.load(f)["version"]

with cliente_local.open(API_LOCAL + "/api/tags", timeout=10) as f:
    modelos = json.load(f)["models"]

modelo_info = next(
    (m for m in modelos if m["name"] == MODELO),
    None
)

assert modelo_info is not None, "Modelo no instalado."

# Información y capacidades del modelo
peticion = Request(
    API_LOCAL + "/api/show",
    data=json.dumps({"model": MODELO}).encode("utf-8"),
    headers={"Content-Type": "application/json"},
    method="POST"
)

with cliente_local.open(peticion, timeout=15) as f:
    detalles = json.load(f)

print("Python:", sys.version.split()[0])
print("Ollama:", version_ollama)
print("Modelo:", MODELO)
print("Identificador:", modelo_info["digest"][:12])
print("Casos verificados:", len(casos))
print("SHA256:", sha_casos)
print("Prompt:", configuracion["prompt_version"])
print("Capacidades:", detalles.get("capabilities"))
print("Thinking:", detalles.get("thinking"))
print("Parámetros:", detalles.get("parameters"))

print("\nArchivos y conexión: correctos")


Python: 3.11.16
Ollama: 0.40.13
Modelo: qwen3.5:4b-q4_K_M
Identificador: d8b0f5e9760c
Casos verificados: 6
SHA256: f73d159bb43cc9c93d2768e997052f4381a2c24bb046c82826b1793c1b8da37d
Prompt: V3
Capacidades: ['completion', 'vision', 'tools', 'thinking']
Thinking: {'values': [False, True], 'default': True}
Parámetros: presence_penalty               1.5
temperature                    1
top_k                          20
top_p                          0.95

Archivos y conexión: correctos


In [2]:
import json
import copy
import time
from uuid import uuid4
from datetime import datetime, timezone
from urllib.request import Request

ARCHIVO = "resultados.jsonl"
CASO_ID = "DIAG-Q35-4B-01"

# Mismo diagnóstico sintético de los modelos anteriores
contenido = (
    "Autores: Autor X y Autor Y.\n"
    "Predicción recibida: favorable (simulada).\n"
    "Modelo predictor y puntuación: no disponibles.\n"
    "[E1] Autor X tiene un artículo sobre bases de datos académicas.\n"
    "[E2] Autor Y tiene un artículo sobre recuperación de información.\n"
    "No se proporcionan datos de colaboración previa."
)

# Recuperar configuración compartida
solicitud = copy.deepcopy(
    configuracion["request_defaults"]
)

solicitud["model"] = MODELO
solicitud["think"] = False

# Mismos parámetros directos de Qwen3.5 9B
solicitud["options"].update({
    "num_ctx": 4096,
    "num_predict": 2048,
    "temperature": 0.7,
    "top_p": 0.8,
    "top_k": 20,
    "min_p": 0.0,
    "presence_penalty": 1.5,
    "repeat_penalty": 1.0
})

# Mantener el prompt V3 original
solicitud["messages"] = [
    {
        "role": "system",
        "content": configuracion["system_prompt"]
    },
    {
        "role": "user",
        "content": configuracion["user_prompt_template"].format(
            case_id=CASO_ID,
            contenido=contenido
        )
    }
]

# Evitar registros duplicados
try:
    with open(ARCHIVO, encoding="utf-8") as f:
        anteriores = [
            json.loads(line)
            for line in f if line.strip()
        ]
except FileNotFoundError:
    anteriores = []

assert not any(
    r["case_id"] == CASO_ID for r in anteriores
), "El diagnóstico ya está registrado."

# Preparar registro
registro = {
    "execution_id": str(uuid4()),
    "case_id": CASO_ID,
    "grupo": "diagnostico",
    "configuracion_modelo": "qwen35_4b_directa_v1",
    "modelo_digest": modelo_info["digest"],
    "ollama_version": version_ollama,
    "sha256_casos": sha_casos,
    "sha256_configuracion": sha_config,
    "fecha_inicio_utc": datetime.now(
        timezone.utc
    ).isoformat(),
    "solicitud": solicitud,
    "reintentos": 0
}

inicio = time.perf_counter()

# Ejecutar en Ollama local
try:
    peticion = Request(
        configuracion["api_url"],
        data=json.dumps(
            solicitud, ensure_ascii=False
        ).encode("utf-8"),
        headers={"Content-Type": "application/json"},
        method="POST"
    )

    with cliente_local.open(peticion, timeout=300) as conexion:
        respuesta = json.load(conexion)

    registro["respuesta_original"] = respuesta

    mensaje = respuesta.get("message", {})

    if respuesta.get("done_reason") == "length":
        registro["estado"] = "truncada"

    elif mensaje.get("thinking"):
        registro["estado"] = "revisar_modo"

    elif (
        respuesta.get("done_reason") == "stop"
        and mensaje.get("content", "").strip()
    ):
        registro["estado"] = "ok"

    else:
        registro["estado"] = "revisar"

except Exception as error:
    registro["estado"] = "error"
    registro["error_tipo"] = type(error).__name__
    registro["error"] = str(error)

registro["tiempo_cliente_s"] = round(
    time.perf_counter() - inicio, 3
)

registro["fecha_fin_utc"] = datetime.now(
    timezone.utc
).isoformat()

# Guardar resultado original
with open(ARCHIVO, "a", encoding="utf-8") as f:
    f.write(json.dumps(registro, ensure_ascii=False) + "\n")

# Mostrar resultados
print("Estado:", registro["estado"])
print("Tiempo cliente:", registro["tiempo_cliente_s"], "s")

if "respuesta_original" in registro:
    r = registro["respuesta_original"]

    print("Tokens entrada:", r.get("prompt_eval_count"))
    print("Tokens generados:", r.get("eval_count"))
    print("Terminación:", r.get("done_reason"))
    print("Razonamiento separado:", bool(
        r.get("message", {}).get("thinking")
    ))

    for campo in (
        "total_duration",
        "load_duration",
        "prompt_eval_duration",
        "eval_duration"
    ):
        valor = r.get(campo)
        if valor is not None:
            print(campo, round(valor / 1e9, 3), "s")

    print("\nRESPUESTA FINAL:\n")
    print(r.get("message", {}).get("content", ""))

else:
    print("ERROR:", registro.get("error"))

print("\nRegistro guardado en resultados.jsonl")


Estado: ok
Tiempo cliente: 17.036 s
Tokens entrada: 376
Tokens generados: 481
Terminación: stop
Razonamiento separado: False
total_duration 17.034 s
load_duration 3.409 s
prompt_eval_duration 7.09 s
eval_duration 6.528 s

RESPUESTA FINAL:

### Evidencia
La única evidencia documentada disponible para el caso DIAG-Q35-4B-01 consiste en dos publicaciones individuales: [E1] atribuida a Autor X sobre bases de datos académicas y [E2] atribuida a Autor Y sobre recuperación de información. No se proporciona ningún dato que indique una colaboración previa entre ambos autores, ni existen registros de proyectos conjuntos o trabajos compartidos hasta la fecha actual.

### Interpretación
Dado que no existe evidencia de colaboración previa, la predicción recibida de "favorable" no puede fundamentarse en un historial real de coautoría. Es importante distinguir que la similitud temática entre los temas de [E1] y [E2] (bases de datos y recuperación de información) no demuestra que los autores hayan tra

In [3]:
import json
import copy
import hashlib
import time
from uuid import uuid4
from datetime import datetime, timezone
from urllib.request import Request, build_opener, ProxyHandler

MODELO = "qwen3.5:4b-q4_K_M"
API = "http://127.0.0.1:11434"
ARCHIVO = "resultados.jsonl"

# 1. Verificar archivos compartidos
with open("../casos_comparacion_v1.json", "rb") as f:
    bytes_casos = f.read()

with open("../configuracion_v3.json", "rb") as f:
    bytes_config = f.read()

casos = json.loads(bytes_casos)
config = json.loads(bytes_config)

sha_casos = hashlib.sha256(bytes_casos).hexdigest()
sha_config = hashlib.sha256(bytes_config).hexdigest()

assert sha_casos == config["cases_sha256"]
assert sha_config == (
    "1c23edad394c7f555f05f2a464440e80cefa1023cb9026593e39eeebacd918af"
)
assert [c["case_id"] for c in casos] == [
    "S-005", "S-006", "S-007",
    "S-008", "S-009", "S-010"
]
assert config["api_url"] == API + "/api/chat"

# 2. Recuperar el diagnóstico exitoso
with open(ARCHIVO, encoding="utf-8") as f:
    anteriores = [
        json.loads(line) for line in f if line.strip()
    ]

diagnostico = next(
    r for r in anteriores
    if r["case_id"] == "DIAG-Q35-4B-01"
)

assert diagnostico["estado"] == "ok"
assert diagnostico["solicitud"]["model"] == MODELO
assert diagnostico["solicitud"]["think"] is False
assert diagnostico["solicitud"]["messages"][0]["content"] == (
    config["system_prompt"]
)

opciones = diagnostico["solicitud"]["options"]

assert opciones["num_ctx"] == 4096
assert opciones["num_predict"] == 2048
assert opciones["temperature"] == 0.7
assert opciones["top_p"] == 0.8
assert opciones["top_k"] == 20
assert opciones["presence_penalty"] == 1.5

# 3. Verificar modelo local
cliente = build_opener(ProxyHandler({}))

with cliente.open(API + "/api/tags", timeout=10) as f:
    modelos = json.load(f)["models"]

modelo_info = next(
    m for m in modelos if m["name"] == MODELO
)

assert modelo_info["digest"] == diagnostico["modelo_digest"]

# Evitar repetir casos ya registrados
registrados = {
    r["case_id"] for r in anteriores
    if r.get("configuracion_modelo") == "qwen35_4b_directa_v1"
    and r.get("grupo") == "comparacion"
}

# 4. Ejecutar los seis casos
with open(ARCHIVO, "a", encoding="utf-8") as archivo:

    for caso in casos:

        case_id = caso["case_id"]

        if case_id in registrados:
            print(case_id, "| ya registrado")
            continue

        # Reutilizar los parámetros del diagnóstico
        solicitud = copy.deepcopy(
            diagnostico["solicitud"]
        )

        # Cambiar únicamente el caso y su evidencia
        solicitud["messages"][1]["content"] = (
            config["user_prompt_template"].format(
                case_id=case_id,
                contenido=caso["contenido"]
            )
        )

        registro = {
            "execution_id": str(uuid4()),
            "case_id": case_id,
            "tipo": caso["tipo"],
            "grupo": "comparacion",
            "configuracion_modelo": "qwen35_4b_directa_v1",
            "prompt_version": "V3",
            "modelo_digest": modelo_info["digest"],
            "ollama_version": diagnostico["ollama_version"],
            "sha256_casos": sha_casos,
            "sha256_configuracion": sha_config,
            "fecha_inicio_utc": datetime.now(
                timezone.utc
            ).isoformat(),
            "solicitud": solicitud,
            "reintentos": 0
        }

        inicio = time.perf_counter()

        try:
            peticion = Request(
                config["api_url"],
                data=json.dumps(
                    solicitud, ensure_ascii=False
                ).encode("utf-8"),
                headers={"Content-Type": "application/json"},
                method="POST"
            )

            with cliente.open(peticion, timeout=360) as conexion:
                respuesta = json.load(conexion)

            registro["respuesta_original"] = respuesta

            mensaje = respuesta.get("message", {})

            if respuesta.get("done_reason") == "length":
                registro["estado"] = "truncada"

            elif mensaje.get("thinking"):
                registro["estado"] = "revisar_modo"

            elif (
                respuesta.get("done_reason") == "stop"
                and mensaje.get("content", "").strip()
            ):
                registro["estado"] = "ok"

            else:
                registro["estado"] = "revisar"

        except Exception as error:
            registro["estado"] = "error"
            registro["error_tipo"] = type(error).__name__
            registro["error"] = str(error)

        registro["tiempo_cliente_s"] = round(
            time.perf_counter() - inicio, 3
        )

        registro["fecha_fin_utc"] = datetime.now(
            timezone.utc
        ).isoformat()

        # Guardar cada ejecución inmediatamente
        archivo.write(
            json.dumps(registro, ensure_ascii=False) + "\n"
        )
        archivo.flush()

        r = registro.get("respuesta_original", {})

        print(
            case_id,
            "|", registro["estado"],
            "|", r.get("eval_count", "-"), "tokens",
            "|", registro["tiempo_cliente_s"], "s"
        )

print("\nArchivo actualizado:", ARCHIVO)


S-005 | ok | 535 tokens | 8.796 s
S-006 | ok | 528 tokens | 8.264 s
S-007 | ok | 612 tokens | 9.392 s
S-008 | ok | 549 tokens | 7.357 s
S-009 | ok | 551 tokens | 8.812 s
S-010 | ok | 533 tokens | 8.326 s

Archivo actualizado: resultados.jsonl


In [4]:
import json
import copy
import time
from uuid import uuid4
from datetime import datetime, timezone
from urllib.request import Request, build_opener, ProxyHandler

ARCHIVO = "resultados.jsonl"
MODELO = "qwen3.5:4b-q4_K_M"
API = "http://127.0.0.1:11434"

# 1. Recuperar el diagnóstico directo
with open(ARCHIVO, encoding="utf-8") as f:
    registros = [
        json.loads(line) for line in f if line.strip()
    ]

assert not any(
    r["case_id"] == "DIAG-Q35-4B-02"
    for r in registros
), "Este diagnóstico ya está registrado."

anterior = next(
    r for r in registros
    if r["case_id"] == "DIAG-Q35-4B-01"
)

assert anterior["estado"] == "ok"
assert anterior["solicitud"]["model"] == MODELO
assert anterior["solicitud"]["think"] is False

# 2. Reutilizar exactamente el mismo caso y prompt
solicitud = copy.deepcopy(anterior["solicitud"])

solicitud["think"] = True

# Misma configuración de Qwen3.5 9B con razonamiento
solicitud["options"].update({
    "num_ctx": 8192,
    "num_predict": 6144,
    "temperature": 1.0,
    "top_p": 0.95,
    "top_k": 20,
    "min_p": 0.0,
    "presence_penalty": 1.5,
    "repeat_penalty": 1.0
})

# 3. Verificar identidad del modelo local
cliente = build_opener(ProxyHandler({}))

with cliente.open(API + "/api/tags", timeout=10) as f:
    instalados = json.load(f)["models"]

modelo_actual = next(
    m for m in instalados if m["name"] == MODELO
)

assert modelo_actual["digest"] == anterior["modelo_digest"]

# 4. Preparar registro
registro = {
    "execution_id": str(uuid4()),
    "case_id": "DIAG-Q35-4B-02",
    "grupo": "diagnostico",
    "configuracion_modelo": "qwen35_4b_thinking_v1",
    "referencia": "DIAG-Q35-4B-01",
    "modelo_digest": modelo_actual["digest"],
    "ollama_version": anterior["ollama_version"],
    "sha256_casos": anterior["sha256_casos"],
    "sha256_configuracion": anterior["sha256_configuracion"],
    "fecha_inicio_utc": datetime.now(
        timezone.utc
    ).isoformat(),
    "solicitud": solicitud,
    "reintentos": 0
}

inicio = time.perf_counter()

# 5. Ejecutar diagnóstico con razonamiento
try:
    peticion = Request(
        API + "/api/chat",
        data=json.dumps(
            solicitud, ensure_ascii=False
        ).encode("utf-8"),
        headers={"Content-Type": "application/json"},
        method="POST"
    )

    with cliente.open(peticion, timeout=360) as conexion:
        respuesta = json.load(conexion)

    registro["respuesta_original"] = respuesta

    mensaje = respuesta.get("message", {})

    if respuesta.get("done_reason") == "length":
        registro["estado"] = "truncada"

    elif (
        respuesta.get("done_reason") == "stop"
        and mensaje.get("content", "").strip()
        and mensaje.get("thinking")
    ):
        registro["estado"] = "ok"

    else:
        registro["estado"] = "revisar"

except Exception as error:
    registro["estado"] = "error"
    registro["error_tipo"] = type(error).__name__
    registro["error"] = str(error)

registro["tiempo_cliente_s"] = round(
    time.perf_counter() - inicio, 3
)

registro["fecha_fin_utc"] = datetime.now(
    timezone.utc
).isoformat()

# 6. Guardar sin sobrescribir resultados anteriores
with open(ARCHIVO, "a", encoding="utf-8") as f:
    f.write(json.dumps(registro, ensure_ascii=False) + "\n")

# 7. Mostrar resultados
print("Estado:", registro["estado"])
print("Tiempo cliente:", registro["tiempo_cliente_s"], "s")

if "respuesta_original" in registro:
    r = registro["respuesta_original"]

    print("Tokens entrada:", r.get("prompt_eval_count"))
    print("Tokens generados:", r.get("eval_count"))
    print("Terminación:", r.get("done_reason"))
    print("Razonamiento separado:", bool(
        r.get("message", {}).get("thinking")
    ))

    print("\nRESPUESTA FINAL:\n")
    print(r.get("message", {}).get("content", ""))

else:
    print("Error:", registro.get("error"))

print("\nRegistro añadido a resultados.jsonl")


Estado: ok
Tiempo cliente: 79.933 s
Tokens entrada: 374
Tokens generados: 5071
Terminación: stop
Razonamiento separado: True

RESPUESTA FINAL:

**Evidencia**
El análisis documentado establece que Autor X posee un artículo sobre bases de datos académicas [E1] y que Autor Y tiene uno sobre recuperación de información [E2]. No se proporcionan registros previos ni datos que indiquen una colaboración entre ambos autores.

**Interpretación**
La evidencia [E1] y [E2] revela una intersección temática en los campos de estudio, lo cual constituye un contexto académico común compatible con la sugerencia de coautoría. Esta similitud permite considerar la pertinencia de una relación profesional desde el ámbito de investigación compartido, diferenciándose así de la prueba directa de trabajos conjuntos previos.

**Limitaciones**
El respaldo de la predicción es parcial ya que no existe documentación sobre colaboración previa ni se conocen los atributos del modelo predictor. No se ha verificado cómo se

In [5]:
import json
import copy
import hashlib
import time
from uuid import uuid4
from datetime import datetime, timezone
from urllib.request import Request, build_opener, ProxyHandler

MODELO = "qwen3.5:4b-q4_K_M"
API = "http://127.0.0.1:11434"
ARCHIVO = "resultados.jsonl"

# 1. Verificar archivos compartidos
with open("../casos_comparacion_v1.json", "rb") as f:
    bytes_casos = f.read()

with open("../configuracion_v3.json", "rb") as f:
    bytes_config = f.read()

casos = json.loads(bytes_casos)
config = json.loads(bytes_config)

sha_casos = hashlib.sha256(bytes_casos).hexdigest()
sha_config = hashlib.sha256(bytes_config).hexdigest()

assert sha_casos == config["cases_sha256"]
assert sha_config == (
    "1c23edad394c7f555f05f2a464440e80cefa1023cb9026593e39eeebacd918af"
)
assert [c["case_id"] for c in casos] == [
    "S-005", "S-006", "S-007",
    "S-008", "S-009", "S-010"
]
assert config["api_url"] == API + "/api/chat"

# 2. Recuperar las ejecuciones anteriores
with open(ARCHIVO, encoding="utf-8") as f:
    anteriores = [
        json.loads(line)
        for line in f if line.strip()
    ]

directos = {
    r["case_id"]: r
    for r in anteriores
    if r.get("configuracion_modelo") == "qwen35_4b_directa_v1"
    and r.get("grupo") == "comparacion"
}

assert set(directos) == {
    c["case_id"] for c in casos
}

# Recuperar diagnóstico con razonamiento
diagnostico = next(
    r for r in anteriores
    if r["case_id"] == "DIAG-Q35-4B-02"
)

assert diagnostico["estado"] == "ok"
assert diagnostico["solicitud"]["think"] is True
assert diagnostico["respuesta_original"]["done_reason"] == "stop"

opciones = copy.deepcopy(
    diagnostico["solicitud"]["options"]
)

assert opciones["num_ctx"] == 8192
assert opciones["num_predict"] == 6144
assert opciones["temperature"] == 1.0
assert opciones["top_p"] == 0.95
assert opciones["top_k"] == 20
assert opciones["presence_penalty"] == 1.5

# 3. Verificar la identidad del modelo
cliente = build_opener(ProxyHandler({}))

with cliente.open(API + "/api/tags", timeout=10) as f:
    instalados = json.load(f)["models"]

modelo_info = next(
    m for m in instalados if m["name"] == MODELO
)

assert modelo_info["digest"] == diagnostico["modelo_digest"]

# Evitar duplicar casos ya ejecutados
registrados = {
    r["case_id"] for r in anteriores
    if r.get("configuracion_modelo") == "qwen35_4b_thinking_v1"
    and r.get("grupo") == "comparacion"
}

# 4. Ejecutar los seis casos
with open(ARCHIVO, "a", encoding="utf-8") as archivo:

    for caso in casos:
        case_id = caso["case_id"]

        if case_id in registrados:
            print(case_id, "| ya registrado")
            continue

        # Recuperar el prompt original del mismo caso
        solicitud = copy.deepcopy(
            directos[case_id]["solicitud"]
        )

        assert solicitud["messages"][0]["content"] == (
            config["system_prompt"]
        )

        assert solicitud["messages"][1]["content"] == (
            config["user_prompt_template"].format(
                case_id=case_id,
                contenido=caso["contenido"]
            )
        )

        # Aplicar configuración de razonamiento
        solicitud["think"] = True
        solicitud["options"] = copy.deepcopy(opciones)

        registro = {
            "execution_id": str(uuid4()),
            "case_id": case_id,
            "tipo": caso["tipo"],
            "grupo": "comparacion",
            "configuracion_modelo": "qwen35_4b_thinking_v1",
            "prompt_version": "V3",
            "modelo_digest": modelo_info["digest"],
            "ollama_version": diagnostico["ollama_version"],
            "sha256_casos": sha_casos,
            "sha256_configuracion": sha_config,
            "fecha_inicio_utc": datetime.now(
                timezone.utc
            ).isoformat(),
            "solicitud": solicitud,
            "reintentos": 0
        }

        inicio = time.perf_counter()

        try:
            peticion = Request(
                config["api_url"],
                data=json.dumps(
                    solicitud, ensure_ascii=False
                ).encode("utf-8"),
                headers={"Content-Type": "application/json"},
                method="POST"
            )

            with cliente.open(peticion, timeout=360) as conexion:
                respuesta = json.load(conexion)

            registro["respuesta_original"] = respuesta

            mensaje = respuesta.get("message", {})

            if respuesta.get("done_reason") == "length":
                registro["estado"] = "truncada"

            elif (
                respuesta.get("done_reason") == "stop"
                and mensaje.get("content", "").strip()
                and mensaje.get("thinking")
            ):
                registro["estado"] = "ok"

            else:
                registro["estado"] = "revisar"

        except Exception as error:
            registro["estado"] = "error"
            registro["error_tipo"] = type(error).__name__
            registro["error"] = str(error)

        registro["tiempo_cliente_s"] = round(
            time.perf_counter() - inicio, 3
        )

        registro["fecha_fin_utc"] = datetime.now(
            timezone.utc
        ).isoformat()

        # Guardar cada resultado inmediatamente
        archivo.write(
            json.dumps(registro, ensure_ascii=False) + "\n"
        )
        archivo.flush()

        r = registro.get("respuesta_original", {})

        print(
            case_id,
            "|", registro["estado"],
            "|", r.get("eval_count", "-"), "tokens",
            "|", registro["tiempo_cliente_s"], "s"
        )

print("\nArchivo actualizado:", ARCHIVO)


S-005 | ok | 3836 tokens | 47.322 s
S-006 | ok | 2992 tokens | 36.237 s
S-007 | ok | 2752 tokens | 30.706 s
S-008 | ok | 4676 tokens | 52.787 s
S-009 | truncada | 6144 tokens | 79.297 s
S-010 | ok | 4013 tokens | 48.781 s

Archivo actualizado: resultados.jsonl


In [6]:
import json
import copy
import time
from uuid import uuid4
from datetime import datetime, timezone
from urllib.request import Request, build_opener, ProxyHandler

ARCHIVO = "resultados.jsonl"
MODELO = "qwen3.5:4b-q4_K_M"
CONFIG_RESCATE = "qwen35_4b_thinking_rescate_8192"

# Recuperar el caso truncado
with open(ARCHIVO, encoding="utf-8") as f:
    registros = [
        json.loads(line) for line in f if line.strip()
    ]

assert not any(
    r.get("configuracion_modelo") == CONFIG_RESCATE
    and r.get("case_id") == "S-009"
    for r in registros
), "La repetición ya está registrada."

anterior = next(
    r for r in registros
    if r["case_id"] == "S-009"
    and r.get("configuracion_modelo") == "qwen35_4b_thinking_v1"
)

assert anterior["estado"] == "truncada"
assert anterior["respuesta_original"]["done_reason"] == "length"
assert anterior["solicitud"]["model"] == MODELO
assert anterior["solicitud"]["think"] is True

# Conservar exactamente el prompt y la evidencia
solicitud = copy.deepcopy(anterior["solicitud"])

# Modificar únicamente contexto y presupuesto de salida
solicitud["options"]["num_ctx"] = 12288
solicitud["options"]["num_predict"] = 8192

# Comprobar la identidad del modelo
cliente = build_opener(ProxyHandler({}))

with cliente.open(
    "http://127.0.0.1:11434/api/tags",
    timeout=10
) as f:
    modelos = json.load(f)["models"]

modelo_actual = next(
    m for m in modelos if m["name"] == MODELO
)

assert modelo_actual["digest"] == anterior["modelo_digest"]

# Preparar registro adicional
registro = {
    "execution_id": str(uuid4()),
    "case_id": "S-009",
    "grupo": "comparacion",
    "configuracion_modelo": CONFIG_RESCATE,
    "referencia_execution_id": anterior["execution_id"],
    "motivo": "Truncamiento con 6144 tokens",
    "prompt_version": "V3",
    "modelo_digest": anterior["modelo_digest"],
    "ollama_version": anterior["ollama_version"],
    "sha256_casos": anterior["sha256_casos"],
    "sha256_configuracion": anterior["sha256_configuracion"],
    "fecha_inicio_utc": datetime.now(
        timezone.utc
    ).isoformat(),
    "solicitud": solicitud,
    "reintentos": 1
}

inicio = time.perf_counter()

try:
    peticion = Request(
        "http://127.0.0.1:11434/api/chat",
        data=json.dumps(
            solicitud, ensure_ascii=False
        ).encode("utf-8"),
        headers={"Content-Type": "application/json"},
        method="POST"
    )

    with cliente.open(peticion, timeout=360) as conexion:
        respuesta = json.load(conexion)

    registro["respuesta_original"] = respuesta

    mensaje = respuesta.get("message", {})

    if respuesta.get("done_reason") == "length":
        registro["estado"] = "truncada"
    elif (
        respuesta.get("done_reason") == "stop"
        and mensaje.get("content", "").strip()
        and mensaje.get("thinking")
    ):
        registro["estado"] = "ok"
    else:
        registro["estado"] = "revisar"

except Exception as error:
    registro["estado"] = "error"
    registro["error_tipo"] = type(error).__name__
    registro["error"] = str(error)

registro["tiempo_cliente_s"] = round(
    time.perf_counter() - inicio, 3
)

registro["fecha_fin_utc"] = datetime.now(
    timezone.utc
).isoformat()

# Añadir el resultado sin borrar el truncamiento anterior
with open(ARCHIVO, "a", encoding="utf-8") as f:
    f.write(json.dumps(registro, ensure_ascii=False) + "\n")

print("Caso:", registro["case_id"])
print("Estado:", registro["estado"])
print("Tiempo cliente:", registro["tiempo_cliente_s"], "s")

if "respuesta_original" in registro:
    r = registro["respuesta_original"]

    print("Tokens entrada:", r.get("prompt_eval_count"))
    print("Tokens generados:", r.get("eval_count"))
    print("Terminación:", r.get("done_reason"))
    print("Razonamiento separado:", bool(
        r.get("message", {}).get("thinking")
    ))

    print("\nRESPUESTA FINAL:\n")
    print(r.get("message", {}).get("content", ""))
else:
    print("Error:", registro.get("error"))

print("\nRegistro añadido a resultados.jsonl")


Caso: S-009
Estado: ok
Tiempo cliente: 68.618 s
Tokens entrada: 396
Tokens generados: 5008
Terminación: stop
Razonamiento separado: True

RESPUESTA FINAL:

Evidencia
La predicción se fundamenta en la información documentada relativa a dos publicaciones individuales: el artículo P-Q01 firmado por Autor Q publicado en 2024 sobre generación aumentada de retiro para respuestas científicas [E1], y el artículo P-R01 elaborado por Autor R en 2025 enfocado en métodos de evaluación para el retiro de evidencia [E2]. Los textos disponibles muestran únicamente a cada autor con su respectivo trabajo, sin registros compartidos entre ambos.

Interpretación
A pesar de que los temas técnicos presentes en los abstractos [E1] y [E2] comparten áreas relevantes como la respuesta científica basada en generación y retiro, los autores aparecen listados individualmente en cada documento. La evidencia no registra ninguna publicación conjunta ni indicaciones de una redacción compartida entre Autor Q y Autor R. D